# LeetCode 499 — The Maze III bằng Python

- Link: https://leetcode.com/problems/the-maze-iii/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Đồ thị, Dijkstra, Heap, Shortest Path với tiêu chí phụ (lexicographic tie-break)

> **Ghi chú Premium:** Đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết phổ biến về bài toán (luật lăn bóng, cách tính khoảng cách, ví dụ, ràng buộc). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** câu chữ, ví dụ và ràng buộc chính xác trên LeetCode.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findShortestWay(self, maze: List[List[int]], ball: List[int], hole: List[int]) -> str:

```

## 1. Mục tiêu học

1. Mô hình hóa "bóng lăn đến khi đụng tường" thành **đồ thị có trọng số**: đỉnh = ô bóng dừng, cạnh = một lần lăn, trọng số = số ô đi qua.
2. Cài đặt hàm **lăn bóng** cẩn thận, xử lý việc **rơi vào lỗ giữa đường**.
3. Dùng **Dijkstra** với khóa so sánh là **cặp `(khoảng cách, chuỗi đường đi)`** để vừa ngắn nhất vừa nhỏ nhất theo thứ tự từ điển.
4. Hiểu vì sao so sánh tuple `(dist, path)` "tương thích" với Dijkstra (không phá vỡ tính đúng).
5. So sánh 3 cách: DFS vét cạn, Bellman-Ford/SPFA, Dijkstra — và kiểm thử chéo trên mê cung ngẫu nhiên.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có một mê cung lưới `maze` (`0` = ô trống, `1` = tường). Một quả bóng ở ô `ball`, một cái lỗ ở ô `hole`. Bóng có thể lăn theo 4 hướng **lên (`u`)**, **xuống (`d`)**, **trái (`l`)**, **phải (`r`)**; nhưng một khi đã lăn, nó **không dừng** cho tới khi đụng tường (hoặc biên mê cung). Nếu trên đường lăn bóng đi qua lỗ, nó **rơi vào lỗ** và dừng ngay.

Khoảng cách của một đường đi = **tổng số ô trống** bóng lăn qua (không tính ô xuất phát, tính ô đích/lỗ).

Hãy trả về chuỗi hướng (ví dụ `"lul"`) giúp bóng rơi vào lỗ với **khoảng cách ngắn nhất**. Nếu có nhiều chuỗi cùng ngắn nhất, trả về chuỗi **nhỏ nhất theo thứ tự từ điển**. Nếu không thể, trả về `"impossible"`.

**Ví dụ 1**

```text
maze = [[0,0,0,0,0],
        [1,1,0,0,1],
        [0,0,0,0,0],
        [0,1,0,0,1],
        [0,1,0,0,0]]
ball = [4,3], hole = [0,1]
Output: "lul"
Giải thích: hai đường ngắn nhất (khoảng cách 6): "lul" (trái → lên → trái) và "ul" (lên → trái).
"lul" < "ul" theo thứ tự từ điển (vì 'l' < 'u').
```

**Ví dụ 2**

```text
Cùng maze, ball = [4,3], hole = [3,0]
Output: "impossible"
```

**Ví dụ 3**

```text
maze = [[0,0,0,0,0,0,0],
        [0,0,1,0,0,1,0],
        [0,0,0,0,1,0,0],
        [0,0,0,0,0,0,1]]
ball = [0,4], hole = [3,5]
Output: "dldr"
```

**Ràng buộc (theo hiểu biết, nên đối chiếu)**

- `m == maze.length`, `n == maze[i].length`, `1 <= m, n <= 100`
- `maze[i][j]` là `0` hoặc `1`.
- `ball` và `hole` đều ở ô trống, và **không trùng nhau**.
- Mọi ô ngoài lưới coi như tường. Mê cung có ít nhất 2 ô trống.

In [1]:
maze1 = [[0, 0, 0, 0, 0],
         [1, 1, 0, 0, 1],
         [0, 0, 0, 0, 0],
         [0, 1, 0, 0, 1],
         [0, 1, 0, 0, 0]]
maze3 = [[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 1, 0],
         [0, 0, 0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0, 0, 1]]

def show(maze, ball=None, hole=None):
    for r, row in enumerate(maze):
        line = ""
        for c, v in enumerate(row):
            if ball and (r, c) == tuple(ball): line += " B"
            elif hole and (r, c) == tuple(hole): line += " O"
            else: line += " #" if v else " ."
        print(line)

show(maze1, [4, 3], [0, 1])

 . O . . .
 # # . . #
 . . . . .
 . # . . #
 . # . B .


## 3. Kiến thức nền

### 3.1. Mô phỏng lăn bóng bằng `while`

Lăn theo hướng `(dr, dc)` từ `(r, c)`: chừng nào ô kế tiếp còn trong lưới và là ô trống thì bước sang, đếm `dist += 1`. **Thêm điều kiện:** nếu ô hiện tại là lỗ thì dừng ngay (rơi vào lỗ).

In [2]:
def roll(maze, r, c, dr, dc, hole):
    m, n = len(maze), len(maze[0])
    dist = 0
    while 0 <= r + dr < m and 0 <= c + dc < n and maze[r + dr][c + dc] == 0:
        r += dr; c += dc; dist += 1
        if (r, c) == hole:
            break
    return r, c, dist

hole = (0, 1)
print("từ (4,3) lăn trái :", roll(maze1, 4, 3, 0, -1, hole))
print("từ (4,3) lăn lên  :", roll(maze1, 4, 3, -1, 0, hole))
print("từ (0,3) lăn trái :", roll(maze1, 0, 3, 0, -1, hole), "<- dừng ở lỗ (0,1) chứ không lăn tới (0,0)")

từ (4,3) lăn trái : (4, 2, 1)
từ (4,3) lăn lên  : (0, 3, 4)
từ (0,3) lăn trái : (0, 1, 2) <- dừng ở lỗ (0,1) chứ không lăn tới (0,0)


### 3.2. Thứ tự từ điển của chuỗi và so sánh tuple

Python so sánh chuỗi theo **thứ tự từ điển**: so ký tự đầu tiên khác nhau; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn. Với 4 hướng: `'d' < 'l' < 'r' < 'u'`.

So sánh **tuple** cũng theo thứ tự từ điển: so phần tử đầu; bằng nhau thì so phần tử thứ hai… Nên `(6, "lul") < (6, "ul") < (7, "a")`.

In [3]:
print(sorted("udlr"))
print("lul" < "ul", "ab" < "abc", (6, "lul") < (6, "ul"), (6, "zzz") < (7, "a"))

['d', 'l', 'r', 'u']
True True True True


### 3.3. Heap với tuple: `heapq`

`heapq` so sánh phần tử bằng `<`, nên đẩy tuple `(dist, path, r, c)` vào heap thì phần tử nhỏ nhất theo `(dist, path)` nổi lên đầu — đúng thứ ta cần cho Dijkstra có tie-break.

In [4]:
import heapq
h = []
for item in [(6, "ul", 0, 1), (6, "lul", 0, 1), (3, "u", 2, 3), (6, "d", 9, 9)]:
    heapq.heappush(h, item)
while h:
    print(heapq.heappop(h))

(3, 'u', 2, 3)
(6, 'd', 9, 9)
(6, 'lul', 0, 1)
(6, 'ul', 0, 1)


### 3.4. Dijkstra cơ bản (nhắc lại)

**Dijkstra** tìm đường đi ngắn nhất từ một đỉnh nguồn trong đồ thị có trọng số **không âm**:
1. `best[src] = 0`, đẩy `(0, src)` vào heap.
2. Lấy đỉnh có khoảng cách nhỏ nhất; nếu đã có giá trị tốt hơn thì bỏ qua (bản ghi cũ).
3. Với mỗi cạnh `(u → v, w)`: nếu `best[u] + w < best[v]` thì cập nhật và đẩy vào heap.

Ví dụ nhỏ trên đồ thị thường:

In [5]:
def dijkstra(graph, src):
    best = {src: 0}
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > best[u]:
            continue
        for v, w in graph.get(u, []):
            if d + w < best.get(v, float("inf")):
                best[v] = d + w
                heapq.heappush(pq, (d + w, v))
    return best

g = {"A": [("B", 4), ("C", 1)], "C": [("B", 2), ("D", 7)], "B": [("D", 1)]}
print(dijkstra(g, "A"))

{'A': 0, 'B': 3, 'C': 1, 'D': 4}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (ball `(4,3)`, hole `(0,1)`):

- Từ `(4,3)`:
  - `l` → lăn tới `(4,2)` (vì `(4,1)` là tường), dist 1.
  - `u` → lăn `(3,3)`, `(2,3)`, `(1,3)`, `(0,3)` → dừng ở `(0,3)`, dist 4.
  - `r` → `(4,4)`, dist 1. `d` → biên, không di chuyển.
- Từ `(0,3)` lăn `l` → `(0,2)`, `(0,1)` = lỗ → rơi. Tổng `4 + 2 = 6`, đường `"ul"`.
- Từ `(4,2)` lăn `u` → `(3,2)`, `(2,2)`, `(1,2)`, `(0,2)` dừng (biên), dist 4; rồi `l` → `(0,1)` lỗ, dist 1. Tổng `1 + 4 + 1 = 6`, đường `"lul"`.

Hai đường cùng khoảng cách 6 → chọn `"lul"` (nhỏ hơn theo từ điển).

Bài học: **không** thể chỉ đếm số lần lăn (BFS theo số bước) — khoảng cách là số ô; và **không** thể chỉ lưu khoảng cách — phải lưu cả chuỗi để phá hòa.

In [6]:
hole = (0, 1)
for name, (dr, dc) in {"d": (1, 0), "l": (0, -1), "r": (0, 1), "u": (-1, 0)}.items():
    print("từ (4,3) lăn", name, "->", roll(maze1, 4, 3, dr, dc, hole))
print("từ (4,2) lăn u ->", roll(maze1, 4, 2, -1, 0, hole))
print("từ (0,2) lăn l ->", roll(maze1, 0, 2, 0, -1, hole))

từ (4,3) lăn d -> (4, 3, 0)
từ (4,3) lăn l -> (4, 2, 1)
từ (4,3) lăn r -> (4, 4, 1)
từ (4,3) lăn u -> (0, 3, 4)
từ (4,2) lăn u -> (0, 2, 4)
từ (0,2) lăn l -> (0, 1, 1)


## 5. Cách 1 — Brute Force: DFS vét mọi đường đi (không lặp điểm dừng)

**Ý tưởng.** Từ vị trí bóng, thử mọi chuỗi lăn. Để không lặp vô hạn, ta không cho đường đi quay lại một **điểm dừng** đã có trên đường (một vòng lặp chỉ làm tăng khoảng cách, nên đường tối ưu không bao giờ chứa vòng). Mỗi khi rơi vào lỗ, so `(dist, path)` với kết quả tốt nhất.

Đây là chuẩn đối chiếu "ngây thơ nhưng dễ tin" cho mê cung nhỏ.

In [7]:
DIRS = [("d", 1, 0), ("l", 0, -1), ("r", 0, 1), ("u", -1, 0)]   # sắp theo chữ cái

def maze3_brute(maze, ball, hole):
    hole = tuple(hole)
    best = [None]

    def dfs(r, c, dist, path, on_path):
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, r, c, dr, dc, hole)
            if d == 0:
                continue
            cand = (dist + d, path + ch)
            if (nr, nc) == hole:
                if best[0] is None or cand < best[0]:
                    best[0] = cand
            elif (nr, nc) not in on_path:
                on_path.add((nr, nc))
                dfs(nr, nc, dist + d, path + ch, on_path)
                on_path.remove((nr, nc))

    start = tuple(ball)
    dfs(start[0], start[1], 0, "", {start})
    return "impossible" if best[0] is None else best[0][1]

print(maze3_brute(maze1, [4, 3], [0, 1]))
print(maze3_brute(maze1, [4, 3], [3, 0]))
print(maze3_brute(maze3, [0, 4], [3, 5]))

lul
impossible
dldr


**Độ phức tạp.** Số đường đi đơn trên đồ thị điểm dừng có thể **mũ** theo số đỉnh. Với mê cung `100 × 100` là bất khả thi.

**Nút thắt.** Ta tính lại cùng một điểm dừng qua vô số đường khác nhau. Nhưng thật ra, để đi tiếp từ một điểm dừng, ta chỉ cần biết **đường tốt nhất** (theo `(dist, path)`) tới điểm đó — đây là tư tưởng đường đi ngắn nhất.

In [8]:
import time, random
random.seed(5)
grid = [[1 if random.random() < 0.2 else 0 for _ in range(6)] for _ in range(6)]
grid[0][0] = 0; grid[5][5] = 0
t = time.perf_counter()
print(maze3_brute(grid, [0, 0], [5, 5]), f"({time.perf_counter() - t:.3f}s cho 6x6)")

impossible (0.000s cho 6x6)


## 6. Bước đột phá

**Insight 1 — Đồ thị điểm dừng.** Đỉnh = ô mà bóng **có thể dừng** (sát tường theo hướng vừa lăn) cộng với lỗ. Cạnh = một lần lăn, trọng số = số ô đi qua (≥ 1). Bài toán thành **đường đi ngắn nhất** từ `ball` tới `hole`.

**Insight 2 — Khóa so sánh là `(dist, path)`.** Ta định nghĩa "đường A tốt hơn đường B" nếu `(distA, pathA) < (distB, pathB)` (tuple). Dijkstra vẫn đúng với khóa này **nếu** việc nối thêm một cạnh giữ thứ tự: `(d1, p1) < (d2, p2)` ⇒ `(d1 + w, p1 + ch) < (d2 + w, p2 + ch)`.
- Nếu `d1 < d2`: hiển nhiên.
- Nếu `d1 == d2` và `p1 < p2`: nếu `p1` **không** là tiền tố của `p2` thì hai chuỗi khác nhau ở một vị trí trước khi hết chuỗi ngắn hơn → thêm `ch` vào cuối không đổi kết quả so sánh. Còn `p1` là tiền tố thật sự của `p2` thì không thể xảy ra khi cùng đích và cùng khoảng cách: phần thêm của `p2` là các lần lăn từ một ô quay về chính ô đó với tổng khoảng cách 0 — vô lý vì mỗi lần lăn ≥ 1 ô.

**Insight 3 — Lỗ là "điểm dừng bắt buộc".** Trong hàm lăn, đi qua lỗ thì dừng. Do đó lỗ là một đỉnh và bóng không bao giờ lăn "xuyên qua" nó.

Quy tắc Dijkstra: **nếu** `(best[u].dist + w, best[u].path + ch) < best[v]` **thì** cập nhật `best[v]` và đẩy vào heap. Lần đầu **lấy ra** lỗ khỏi heap → đáp án.

In [9]:
# Minh họa Insight 2: nối thêm cùng một ký tự giữ nguyên thứ tự khi không có quan hệ tiền tố
pairs = [((6, "lul"), (6, "ul")), ((5, "dr"), (5, "ld")), ((4, "u"), (6, "d"))]
for a, b in pairs:
    a2 = (a[0] + 2, a[1] + "r"); b2 = (b[0] + 2, b[1] + "r")
    print(a, "<", b, ":", a < b, "| sau khi nối:", a2 < b2)

(6, 'lul') < (6, 'ul') : True | sau khi nối: True
(5, 'dr') < (5, 'ld') : True | sau khi nối: True
(4, 'u') < (6, 'd') : True | sau khi nối: True


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bellman-Ford / SPFA (hàng đợi, nới lỏng đến khi ổn định)

Giữ `best[(r, c)] = (dist, path)`. Dùng hàng đợi FIFO: lấy một đỉnh, thử 4 hướng lăn, nếu tìm được khóa tốt hơn cho đỉnh đích thì cập nhật và đưa đỉnh đó vào hàng đợi (nếu chưa có). Dừng khi hàng đợi rỗng. Đáp án là `best[hole]`.

SPFA đúng với trọng số không âm (và cả âm không có chu trình âm), nhưng có thể xử lý một đỉnh nhiều lần.

In [10]:
from collections import deque

def maze3_spfa(maze, ball, hole):
    hole = tuple(hole)
    start = tuple(ball)
    best = {start: (0, "")}
    q = deque([start]); in_q = {start}
    while q:
        u = q.popleft(); in_q.discard(u)
        if u == hole:
            continue                      # bóng đã rơi, không lăn tiếp từ lỗ
        d0, p0 = best[u]
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, u[0], u[1], dr, dc, hole)
            if d == 0:
                continue
            cand = (d0 + d, p0 + ch)
            v = (nr, nc)
            if v not in best or cand < best[v]:
                best[v] = cand
                if v not in in_q:
                    q.append(v); in_q.add(v)
    return best[hole][1] if hole in best else "impossible"

print(maze3_spfa(maze1, [4, 3], [0, 1]), maze3_spfa(maze1, [4, 3], [3, 0]), maze3_spfa(maze3, [0, 4], [3, 5]))

lul impossible dldr


**Code nộp LeetCode (Cách 2):**

In [11]:
from typing import List

class SolutionSPFA:
    def findShortestWay(self, maze: List[List[int]], ball: List[int], hole: List[int]) -> str:
        return maze3_spfa(maze, ball, hole)

print(SolutionSPFA().findShortestWay(maze1, [4, 3], [0, 1]))

lul


**Độ phức tạp Cách 2:** mỗi lần xử lý đỉnh tốn `O(4 · max(m, n))` để lăn. Trường hợp xấu một đỉnh vào hàng đợi nhiều lần → `O(V · E · max(m,n))` lý thuyết, nhưng thực tế thường nhanh. Space `O(m·n · L)` với `L` là độ dài chuỗi đường đi được lưu.

### 7.2. Cách 3 — Dijkstra với heap `(dist, path, r, c)` (lời giải chính)

In [12]:
def maze3_dijkstra(maze, ball, hole):
    hole = tuple(hole)
    start = tuple(ball)
    best = {start: (0, "")}
    pq = [(0, "", start[0], start[1])]
    while pq:
        dist, path, r, c = heapq.heappop(pq)
        if (dist, path) > best.get((r, c), (float("inf"), "")):
            continue                              # bản ghi cũ
        if (r, c) == hole:
            return path                           # lấy ra lỗ lần đầu -> tối ưu
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, r, c, dr, dc, hole)
            if d == 0:
                continue
            cand = (dist + d, path + ch)
            if (nr, nc) not in best or cand < best[(nr, nc)]:
                best[(nr, nc)] = cand
                heapq.heappush(pq, (cand[0], cand[1], nr, nc))
    return "impossible"

print(maze3_dijkstra(maze1, [4, 3], [0, 1]), maze3_dijkstra(maze1, [4, 3], [3, 0]), maze3_dijkstra(maze3, [0, 4], [3, 5]))
assert maze3_dijkstra(maze1, [4, 3], [0, 1]) == "lul"

lul impossible dldr


**Code nộp LeetCode (Cách 3):** xem mục 9 (bản tự chứa).

**Độ phức tạp Cách 3:** mỗi đỉnh (≤ `m·n`) được lấy ra hữu hiệu một lần; mỗi lần lăn 4 hướng tốn `O(m + n)`; heap `O(log(mn))`; so sánh chuỗi `O(L)`. Tổng khoảng `O(mn · (m + n + L log(mn)))`. Space `O(mn · L)`.

### 7.3. Cách 4 — Dijkstra "hai pha" (không lưu chuỗi trong heap)

Ở Cách 3, mỗi bản ghi trong heap mang theo cả chuỗi đường đi → tốn bộ nhớ `O(L)` mỗi bản ghi và mỗi phép so sánh có thể tốn `O(L)`. Cách 4 tách làm hai pha:

1. **Xây đồ thị điểm dừng** (BFS từ `ball`, mỗi đỉnh tối đa 4 cạnh `(đích, trọng số, ký tự)`).
2. **Tính `T[v]` = khoảng cách ngắn nhất từ `v` tới lỗ** bằng Dijkstra trên đồ thị **đảo cạnh**, chỉ dùng số nguyên.
3. **Dựng chuỗi tham lam:** đứng ở `ball`; trong các cạnh `(v → w, d, ch)` thỏa `d + T[w] == T[v]` (cạnh "chặt" — nằm trên một đường ngắn nhất), chọn `ch` nhỏ nhất; đi theo cạnh đó; lặp tới khi tới lỗ.

Tại sao tham lam theo ký tự nhỏ nhất là đúng? Mọi đường ngắn nhất đi trên các cạnh "chặt" (`d + T[w] == T[v]`); chọn ký tự đầu nhỏ nhất có thể là bắt buộc để chuỗi nhỏ nhất, rồi đệ quy cho phần còn lại. Mỗi đỉnh chỉ có tối đa 1 cạnh mỗi hướng nên ký tự quyết định duy nhất cạnh.

In [13]:
def maze3_two_phase(maze, ball, hole):
    hole = tuple(hole); start = tuple(ball)
    # Xây đồ thị điểm dừng bằng BFS từ start
    adj = {}
    q = deque([start]); seen = {start}
    while q:
        u = q.popleft()
        adj[u] = []
        if u == hole:
            continue
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, u[0], u[1], dr, dc, hole)
            if d:
                adj[u].append(((nr, nc), d, ch))
                if (nr, nc) not in seen:
                    seen.add((nr, nc)); q.append((nr, nc))
    if hole not in seen:
        return "impossible"
    # T[v] = khoảng cách ngắn nhất từ v tới hole (Dijkstra trên đồ thị đảo cạnh)
    radj = {}
    for u, lst in adj.items():
        for v, d, ch in lst:
            radj.setdefault(v, []).append((u, d))
    T = {hole: 0}; pq = [(0, hole)]
    while pq:
        d0, v = heapq.heappop(pq)
        if d0 > T[v]:
            continue
        for u, d in radj.get(v, []):
            if d0 + d < T.get(u, float("inf")):
                T[u] = d0 + d; heapq.heappush(pq, (d0 + d, u))
    # Đi tham lam trên các cạnh chặt, chọn ký tự nhỏ nhất
    path, u = "", start
    while u != hole:
        options = [(ch, v) for v, d, ch in adj[u] if v in T and d + T[v] == T[u]]
        ch, u = min(options)
        path += ch
    return path

print(maze3_two_phase(maze1, [4, 3], [0, 1]), maze3_two_phase(maze1, [4, 3], [3, 0]), maze3_two_phase(maze3, [0, 4], [3, 5]))

lul impossible dldr


**Code nộp LeetCode (Cách 4):**

In [14]:
class SolutionTwoPhase:
    def findShortestWay(self, maze: List[List[int]], ball: List[int], hole: List[int]) -> str:
        return maze3_two_phase(maze, ball, hole)

print(SolutionTwoPhase().findShortestWay(maze3, [0, 4], [3, 5]))

dldr


**Độ phức tạp Cách 4:** xây đồ thị `O(mn · (m+n))`, Dijkstra `O(mn log(mn))`, dựng chuỗi `O(L)`. Không phải so sánh chuỗi trong heap → bộ nhớ `O(mn)` thay vì `O(mn · L)`.

## 8. TRACE từng bước

Trace **Cách 3** trên ví dụ 1. Mỗi dòng: bản ghi lấy ra khỏi heap `(dist, path, ô)`, và các cập nhật `best` sinh ra.

| bước | lấy ra | cập nhật |
|---|---|---|
| 1 | (0, "", (4,3)) | (4,2)←(1,"l"), (4,4)←(1,"r"), (0,3)←(4,"u") |
| 2 | (1, "l", (4,2)) | (0,2)←(5,"lu") |
| 3 | (1, "r", (4,4)) | (2,4)... |
| … | … | … |
| k | (6, "lul", (0,1)) | **lỗ → trả về "lul"** |

In [15]:
def trace_dijkstra(maze, ball, hole):
    hole = tuple(hole); start = tuple(ball)
    best = {start: (0, "")}
    pq = [(0, "", start[0], start[1])]
    step = 0
    while pq:
        dist, path, r, c = heapq.heappop(pq)
        if (dist, path) > best[(r, c)]:
            print(f"   bỏ bản ghi cũ ({dist}, {path!r}, {(r, c)})")
            continue
        step += 1
        if (r, c) == hole:
            print(f"bước {step}: lấy ra lỗ {(r, c)} với ({dist}, {path!r}) -> TRẢ VỀ")
            return path
        updates = []
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, r, c, dr, dc, hole)
            if d == 0:
                continue
            cand = (dist + d, path + ch)
            if (nr, nc) not in best or cand < best[(nr, nc)]:
                best[(nr, nc)] = cand
                heapq.heappush(pq, (cand[0], cand[1], nr, nc))
                updates.append(f"{(nr, nc)}<-{cand}")
        print(f"bước {step}: lấy ra ({dist}, {path!r}, {(r, c)}) | cập nhật: {', '.join(updates) or '-'}")
    print("impossible")
    return "impossible"

trace_dijkstra(maze1, [4, 3], [0, 1])

bước 1: lấy ra (0, '', (4, 3)) | cập nhật: (4, 2)<-(1, 'l'), (4, 4)<-(1, 'r'), (0, 3)<-(4, 'u')
bước 2: lấy ra (1, 'l', (4, 2)) | cập nhật: (0, 2)<-(5, 'lu')
bước 3: lấy ra (1, 'r', (4, 4)) | cập nhật: -
bước 4: lấy ra (4, 'u', (0, 3)) | cập nhật: (0, 1)<-(6, 'ul'), (0, 4)<-(5, 'ur')
bước 5: lấy ra (5, 'lu', (0, 2)) | cập nhật: (0, 1)<-(6, 'lul')
bước 6: lấy ra (5, 'ur', (0, 4)) | cập nhật: -
bước 7: lấy ra lỗ (0, 1) với (6, 'lul') -> TRẢ VỀ


'lul'

Trace ví dụ 3 để thấy đường `"dldr"`:

In [16]:
show(maze3, [0, 4], [3, 5])
trace_dijkstra(maze3, [0, 4], [3, 5])

 . . . . B . .
 . . # . . # .
 . . . . # . .
 . . . . . O #
bước 1: lấy ra (0, '', (0, 4)) | cập nhật: (1, 4)<-(1, 'd'), (0, 0)<-(4, 'l'), (0, 6)<-(2, 'r')
bước 2: lấy ra (1, 'd', (1, 4)) | cập nhật: (1, 3)<-(2, 'dl')
bước 3: lấy ra (2, 'dl', (1, 3)) | cập nhật: (3, 3)<-(4, 'dld'), (0, 3)<-(3, 'dlu')
bước 4: lấy ra (2, 'r', (0, 6)) | cập nhật: (2, 6)<-(4, 'rd')
bước 5: lấy ra (3, 'dlu', (0, 3)) | cập nhật: -
bước 6: lấy ra (4, 'dld', (3, 3)) | cập nhật: (3, 0)<-(7, 'dldl'), (3, 5)<-(6, 'dldr')
bước 7: lấy ra (4, 'l', (0, 0)) | cập nhật: -
bước 8: lấy ra (4, 'rd', (2, 6)) | cập nhật: (2, 5)<-(5, 'rdl')
bước 9: lấy ra (5, 'rdl', (2, 5)) | cập nhật: -
bước 10: lấy ra lỗ (3, 5) với (6, 'dldr') -> TRẢ VỀ


'dldr'

## 9. Code nộp LeetCode

In [17]:
import heapq
from typing import List

class Solution:
    def findShortestWay(self, maze: List[List[int]], ball: List[int], hole: List[int]) -> str:
        m, n = len(maze), len(maze[0])
        hr, hc = hole
        dirs = [("d", 1, 0), ("l", 0, -1), ("r", 0, 1), ("u", -1, 0)]

        def roll(r, c, dr, dc):
            dist = 0
            while 0 <= r + dr < m and 0 <= c + dc < n and maze[r + dr][c + dc] == 0:
                r += dr; c += dc; dist += 1
                if r == hr and c == hc:          # rơi vào lỗ giữa đường
                    break
            return r, c, dist

        best = {(ball[0], ball[1]): (0, "")}
        pq = [(0, "", ball[0], ball[1])]
        while pq:
            dist, path, r, c = heapq.heappop(pq)
            if (dist, path) > best[(r, c)]:
                continue
            if r == hr and c == hc:
                return path
            for ch, dr, dc in dirs:
                nr, nc, d = roll(r, c, dr, dc)
                if d == 0:
                    continue
                cand = (dist + d, path + ch)
                if (nr, nc) not in best or cand < best[(nr, nc)]:
                    best[(nr, nc)] = cand
                    heapq.heappush(pq, (cand[0], cand[1], nr, nc))
        return "impossible"

sol = Solution()
assert sol.findShortestWay(maze1, [4, 3], [0, 1]) == "lul"
assert sol.findShortestWay(maze1, [4, 3], [3, 0]) == "impossible"
assert sol.findShortestWay(maze3, [0, 4], [3, 5]) == "dldr"
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

1. **Mô hình đúng.** Mọi chuỗi lệnh có ích tương ứng một đường đi trên đồ thị điểm dừng (mỗi lệnh làm bóng di chuyển là một cạnh). Lệnh không làm bóng di chuyển (`d = 0`) chỉ làm chuỗi dài thêm mà không đổi vị trí, nên không bao giờ xuất hiện trong đáp án; ta bỏ các cạnh đó. Hàm `roll` dừng tại lỗ khi đi qua, đúng luật đề.
2. **Thứ tự toàn phần tương thích.** Khóa `(dist, path)` là thứ tự toàn phần; mọi cạnh có `w >= 1`; và theo Insight 2, nối thêm cạnh giữ thứ tự giữa hai đường **tới cùng một đỉnh**. Đây chính là hai tính chất Dijkstra cần: (a) khóa không giảm dọc theo đường đi (`dist` tăng ngặt), (b) "tiền tố tối ưu" — tiền tố của đường tối ưu tới `v` là đường tối ưu tới đỉnh trung gian (nếu không, thay tiền tố bằng đường tốt hơn sẽ được đường tốt hơn tới `v` nhờ (b) của Insight 2).
3. **Lấy ra lần đầu là tối ưu.** Khi lỗ được lấy ra với khóa `K`, mọi bản ghi còn trong heap có khóa `>= K`, và mọi đường đi khác tới lỗ phải đi qua một đỉnh trên "biên" có khóa `>= K` rồi cộng thêm cạnh dương (dist tăng ngặt) → không thể nhỏ hơn `K`.
4. **"impossible".** Nếu heap cạn mà chưa lấy ra lỗ thì lỗ không đạt tới được.

## 11. Phân tích Time/Space

Gọi `V <= m·n` là số điểm dừng, mỗi đỉnh có ≤ 4 cạnh, mỗi lần lăn `O(m + n)`, `L` là độ dài chuỗi (≤ số đỉnh).

| Cách | Time | Space |
|---|---|---|
| 1. DFS vét cạn | mũ | `O(V)` stack |
| 2. SPFA | `O(V · E · (m+n))` xấu nhất | `O(V · L)` |
| 3. Dijkstra (tuple) | `O(V (m+n) + V log V · L)` | `O(V · L)` |
| 4. Dijkstra hai pha | `O(V (m+n) + V log V)` | `O(V)` |

Đo trên mê cung ngẫu nhiên lớn:

In [18]:
random.seed(499)
M = N = 60
big = [[1 if random.random() < 0.25 else 0 for _ in range(N)] for _ in range(M)]
big[0][0] = 0; big[M - 1][N - 1] = 0
for name, f in [("spfa", maze3_spfa), ("dijkstra", maze3_dijkstra), ("two_phase", maze3_two_phase), ("Solution", Solution().findShortestWay)]:
    t = time.perf_counter()
    ans = f(big, [0, 0], [M - 1, N - 1])
    print(f"{name:<10}: {time.perf_counter() - t:.3f}s -> {ans[:30]}{'...' if len(ans) > 30 else ''}")

spfa      : 0.000s -> impossible
dijkstra  : 0.000s -> impossible
two_phase : 0.000s -> impossible
Solution  : 0.000s -> impossible


## 12. Bộ test

In [19]:
methods = {
    "brute": maze3_brute,
    "spfa": maze3_spfa,
    "dijkstra": maze3_dijkstra,
    "two_phase": maze3_two_phase,
    "SolSPFA": SolutionSPFA().findShortestWay,
    "SolTwoPhase": SolutionTwoPhase().findShortestWay,
    "Solution": Solution().findShortestWay,
}
cases = [
    (maze1, [4, 3], [0, 1], "lul"),
    (maze1, [4, 3], [3, 0], "impossible"),
    (maze3, [0, 4], [3, 5], "dldr"),
    ([[0, 0]], [0, 0], [0, 1], "r"),                         # mê cung 1x2
    ([[0], [0], [0]], [2, 0], [1, 0], "u"),                  # rơi vào lỗ giữa đường
    ([[0, 1, 0]], [0, 0], [0, 2], "impossible"),             # bị tường chắn
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], [0, 0], [1, 1], "impossible"),   # lỗ ở giữa, không thể dừng/đi qua
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], [0, 0], [2, 2], "dr"),           # "dr" và "rd" cùng 4 -> "dr"
]
for maze, b, h, exp in cases:
    for name, f in methods.items():
        got = f(maze, b, h)
        assert got == exp, (name, maze, b, h, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


Kiểm tra lại case "lỗ ở giữa lưới 3×3 trống": bóng ở góc chỉ lăn dọc theo biên (luôn đụng biên mới dừng), nên không bao giờ đi qua ô `(1,1)` → `"impossible"`. Case cuối: cả `"dr"` lẫn `"rd"` đều dài 4 → chọn `"dr"`.

In [20]:
def random_maze(m, n, p):
    g = [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(m)]
    cells = [(r, c) for r in range(m) for c in range(n)]
    random.shuffle(cells)
    b, h = cells[0], cells[1]
    g[b[0]][b[1]] = 0; g[h[0]][h[1]] = 0
    return g, list(b), list(h)

random.seed(2026)
cnt_possible = 0
for trial in range(500):
    m, n = random.randint(1, 6), random.randint(2, 6)
    g, b, h = random_maze(m, n, random.choice([0.0, 0.15, 0.3]))
    exp = maze3_brute(g, b, h)
    cnt_possible += exp != "impossible"
    for name, f in methods.items():
        got = f(g, b, h)
        assert got == exp, (name, g, b, h, got, exp)
print(f"500 mê cung ngẫu nhiên (seed 2026) khớp brute force; {cnt_possible} có đường đi")

500 mê cung ngẫu nhiên (seed 2026) khớp brute force; 441 có đường đi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra lỗ khi đang lăn

**Code sai:** lăn tới khi đụng tường rồi mới kiểm tra ô dừng có phải lỗ không.

**Vì sao sai:** bóng lẽ ra rơi vào lỗ **giữa đường** — code sai thì lăn xuyên qua lỗ.

In [21]:
def roll_wrong(maze, r, c, dr, dc, hole):
    m, n = len(maze), len(maze[0]); dist = 0
    while 0 <= r + dr < m and 0 <= c + dc < n and maze[r + dr][c + dc] == 0:
        r += dr; c += dc; dist += 1
    return r, c, dist

print("sai :", roll_wrong([[0], [0], [0]], 2, 0, -1, 0, (1, 0)), "-> lăn xuyên qua lỗ (1,0)")
print("đúng:", roll([[0], [0], [0]], 2, 0, -1, 0, (1, 0)))

sai : (0, 0, 2) -> lăn xuyên qua lỗ (1,0)
đúng: (1, 0, 1)


**Code đúng:** `if (r, c) == hole: break` **bên trong** vòng lăn.

### Lỗi 2 — Chỉ so khoảng cách, bỏ qua chuỗi khi bằng nhau

**Code sai:** `if dist + d < best_dist[v]: ...` → khi bằng nhau không cập nhật, chuỗi trả về phụ thuộc thứ tự duyệt, có thể không phải nhỏ nhất.

In [22]:
def maze3_dist_only(maze, ball, hole):
    hole = tuple(hole); start = tuple(ball)
    best = {start: 0}; pathof = {start: ""}
    pq = [(0, start)]
    order = [("u", -1, 0), ("r", 0, 1), ("l", 0, -1), ("d", 1, 0)]     # thứ tự duyệt khác chữ cái
    while pq:
        dist, (r, c) = heapq.heappop(pq)
        if dist > best[(r, c)]:
            continue
        if (r, c) == hole:
            return pathof[(r, c)]
        for ch, dr, dc in order:
            nr, nc, d = roll(maze, r, c, dr, dc, hole)
            if d and dist + d < best.get((nr, nc), float("inf")):     # SAI: không xét hòa
                best[(nr, nc)] = dist + d; pathof[(nr, nc)] = pathof[(r, c)] + ch
                heapq.heappush(pq, (dist + d, (nr, nc)))
    return "impossible"

print("chỉ so dist:", maze3_dist_only(maze1, [4, 3], [0, 1]), "| đúng:", Solution().findShortestWay(maze1, [4, 3], [0, 1]))

chỉ so dist: ul | đúng: lul


**Code đúng:** so sánh tuple `(dist, path)`.

### Lỗi 3 — Trả về ngay khi **đẩy** lỗ vào heap

**Code sai:** `if (nr, nc) == hole: return path + ch` ngay lúc sinh ra.

**Vì sao sai:** có thể vẫn còn đường khác tới lỗ ngắn hơn hoặc bằng nhưng nhỏ hơn theo từ điển, chưa được xét.

In [23]:
def maze3_early(maze, ball, hole):
    hole = tuple(hole); start = tuple(ball)
    best = {start: (0, "")}; pq = [(0, "", start[0], start[1])]
    while pq:
        dist, path, r, c = heapq.heappop(pq)
        if (dist, path) > best[(r, c)]:
            continue
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, r, c, dr, dc, hole)
            if not d:
                continue
            if (nr, nc) == hole:
                return path + ch                          # SAI: trả về quá sớm
            cand = (dist + d, path + ch)
            if (nr, nc) not in best or cand < best[(nr, nc)]:
                best[(nr, nc)] = cand; heapq.heappush(pq, (*cand, nr, nc))
    return "impossible"

print("trả về sớm:", maze3_early(maze1, [4, 3], [0, 1]), "| đúng:", Solution().findShortestWay(maze1, [4, 3], [0, 1]))

trả về sớm: ul | đúng: lul


**Code đúng:** chỉ trả về khi **lấy ra** lỗ khỏi heap.

### Lỗi 4 — Tính khoảng cách là số lần lăn

BFS theo số lần lăn cho "ít lệnh nhất", không phải "ít ô nhất". Ví dụ 1: `"ul"` chỉ 2 lệnh nhưng cùng khoảng cách 6 với `"lul"`; trên mê cung khác, đường ít lệnh có thể dài hơn nhiều ô. Phải dùng Dijkstra với trọng số = số ô.

## 14. Mẹo & nhận dạng pattern

- "Lăn/trượt tới khi đụng tường" → đồ thị **điểm dừng**, cạnh có trọng số = số ô (The Maze I/II/III, Ice sliding puzzles).
- "Ngắn nhất, hòa thì nhỏ nhất theo từ điển" → Dijkstra với khóa **tuple** `(cost, path)`; kiểm tra tính chất "nối thêm giữ thứ tự".
- Trả về đáp án khi **lấy ra** khỏi heap, không phải khi đẩy vào.
- Có đích "hút" giữa đường (lỗ) → xử lý ngay trong hàm di chuyển.
- Sắp xếp danh sách hướng theo chữ cái (`d, l, r, u`) giúp debug dễ nhưng không thay thế được so sánh chuỗi.

## 15. Biến thể

1. **The Maze (LC 490):** chỉ hỏi có dừng được ở đích không → BFS/DFS trên điểm dừng.
2. **The Maze II (LC 505):** khoảng cách ngắn nhất để **dừng** tại đích (không có lỗ hút) → Dijkstra theo `dist`.
3. **Đếm số đường ngắn nhất** tới lỗ → Dijkstra + đếm.
4. **Trả về số lệnh ít nhất** (không quan tâm số ô) → BFS theo lớp.

Demo biến thể 2 (The Maze II):

In [24]:
def maze2_shortest(maze, start, dest):
    m, n = len(maze), len(maze[0])
    best = {tuple(start): 0}; pq = [(0, start[0], start[1])]
    while pq:
        d0, r, c = heapq.heappop(pq)
        if d0 > best[(r, c)]:
            continue
        if [r, c] == list(dest):
            return d0
        for _, dr, dc in DIRS:
            nr, nc, d = r, c, 0
            while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                nr += dr; nc += dc; d += 1
            if d and d0 + d < best.get((nr, nc), float("inf")):
                best[(nr, nc)] = d0 + d; heapq.heappush(pq, (d0 + d, nr, nc))
    return -1

mz = [[0, 0, 1, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 1, 0], [1, 1, 0, 1, 1], [0, 0, 0, 0, 0]]
print(maze2_shortest(mz, [0, 4], [4, 4]))   # ví dụ quen thuộc của LC 505: 12
assert maze2_shortest(mz, [0, 4], [4, 4]) == 12

12


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS vét cạn | Mọi đường không lặp điểm dừng | mũ | `O(V)` | Đối chiếu, mê cung nhỏ |
| 2. SPFA | Nới lỏng bằng hàng đợi tới khi ổn định | xấu nhất lớn | `O(V·L)` | Dễ viết, chấp nhận được |
| 3. Dijkstra tuple | Heap theo `(dist, path)` | `O(V(m+n) + V log V · L)` | `O(V·L)` | Lời giải chuẩn |
| 4. Dijkstra hai pha | Khoảng cách tới lỗ + đi tham lam trên cạnh chặt | `O(V(m+n) + V log V)` | `O(V)` | Khi chuỗi dài, muốn tiết kiệm bộ nhớ |

## 17. Bài tập

**Bài 1 (đoán output).** `roll(maze1, 0, 4, 0, -1, (0, 1))` trả về gì?

**Bài 2 (tính tay).** Trong mê cung `3×3` toàn ô trống, ball `(0,0)`, hole `(0,2)`. Đáp án là gì? Khoảng cách bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thiện điều kiện cập nhật trong Dijkstra.

In [25]:
def should_update_todo(best, v, cand):
    """best: dict ô -> (dist, path). Trả True nếu cand tốt hơn kết quả hiện có của v."""
    # TODO: True nếu v chưa có trong best hoặc cand < best[v] (so sánh tuple)
    raise NotImplementedError

# Không gọi khi chạy notebook.

**Bài 4 (tự viết lại).** Viết lại `Solution` không nhìn mục 9.

**Bài 5 (mini challenge).** Viết `count_shortest_ways(maze, ball, hole)` trả về **số chuỗi lệnh khác nhau** đạt khoảng cách ngắn nhất (0 nếu không thể).

In [26]:
def count_shortest_ways_todo(maze, ball, hole):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Lăn từng ô một.
- Mức 2: Bài 1: từ `(0,4)` lăn trái qua `(0,3)`, `(0,2)`, `(0,1)` = lỗ.
- Mức 3: Bài 2: `"r"`, khoảng cách 2.

**Bài 3**
- Mức 1: `v not in best`.
- Mức 2: Python so sánh tuple sẵn.
- Mức 3: `return v not in best or cand < best[v]`.

**Bài 4**
- Mức 1: Hàm `roll` có kiểm tra lỗ.
- Mức 2: Heap chứa `(dist, path, r, c)`.
- Mức 3: Trả về khi lấy ra lỗ.

**Bài 5**
- Mức 1: Dijkstra theo `dist` + mảng đếm `ways`.
- Mức 2: Khi tìm được `dist` nhỏ hơn: `ways[v] = ways[u]`; bằng: `ways[v] += ways[u]`.
- Mức 3: Cẩn thận: lỗ không có cạnh đi ra; mỗi chuỗi lệnh tương ứng đúng một đường trên đồ thị.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
assert roll(maze1, 0, 4, 0, -1, (0, 1)) == (0, 1, 3)
assert Solution().findShortestWay([[0] * 3 for _ in range(3)], [0, 0], [0, 2]) == "r"

# Bài 3
def should_update(best, v, cand):
    return v not in best or cand < best[v]
assert should_update({}, (0, 0), (1, "a")) and not should_update({(0, 0): (1, "a")}, (0, 0), (1, "b"))

# Bài 4
class SolutionEx4:
    def findShortestWay(self, maze, ball, hole):
        h = tuple(hole)
        best = {tuple(ball): (0, "")}
        pq = [(0, "", *ball)]
        while pq:
            d0, p0, r, c = heapq.heappop(pq)
            if (d0, p0) > best[(r, c)]:
                continue
            if (r, c) == h:
                return p0
            for ch, dr, dc in DIRS:
                nr, nc, d = roll(maze, r, c, dr, dc, h)
                cand = (d0 + d, p0 + ch)
                if d and should_update(best, (nr, nc), cand):
                    best[(nr, nc)] = cand; heapq.heappush(pq, (*cand, nr, nc))
        return "impossible"

random.seed(77)
for _ in range(300):
    g, b, h = random_maze(random.randint(1, 6), random.randint(2, 6), 0.2)
    assert SolutionEx4().findShortestWay(g, b, h) == maze3_brute(g, b, h)
print("Bài 1–4 OK")

Bài 1–4 OK


In [28]:
# Bài 5
def count_shortest_ways(maze, ball, hole):
    h = tuple(hole); s = tuple(ball)
    dist = {s: 0}; ways = {s: 1}; pq = [(0, s)]; done = set()
    while pq:
        d0, u = heapq.heappop(pq)
        if u in done or d0 > dist[u]:
            continue
        done.add(u)
        if u == h:
            continue
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, u[0], u[1], dr, dc, h)
            if not d:
                continue
            v = (nr, nc)
            if d0 + d < dist.get(v, float("inf")):
                dist[v] = d0 + d; ways[v] = ways[u]; heapq.heappush(pq, (d0 + d, v))
            elif d0 + d == dist[v]:
                ways[v] += ways[u]
    return ways.get(h, 0)

def count_shortest_brute(maze, ball, hole):
    h = tuple(hole); found = []
    def dfs(r, c, dist, on_path):
        for ch, dr, dc in DIRS:
            nr, nc, d = roll(maze, r, c, dr, dc, h)
            if not d:
                continue
            if (nr, nc) == h:
                found.append(dist + d)
            elif (nr, nc) not in on_path:
                on_path.add((nr, nc)); dfs(nr, nc, dist + d, on_path); on_path.remove((nr, nc))
    dfs(ball[0], ball[1], 0, {tuple(ball)})
    return found.count(min(found)) if found else 0

print("Ví dụ 1 có", count_shortest_ways(maze1, [4, 3], [0, 1]), "đường ngắn nhất")
assert count_shortest_ways(maze1, [4, 3], [0, 1]) == 2
random.seed(88)
for _ in range(300):
    g, b, h = random_maze(random.randint(1, 5), random.randint(2, 5), 0.2)
    assert count_shortest_ways(g, b, h) == count_shortest_brute(g, b, h), (g, b, h)
print("Bài 5 OK")

Ví dụ 1 có 2 đường ngắn nhất
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Dijkstra với tie-break chuỗi:**

```python
best = {start: (0, "")}
pq = [(0, "", *start)]
while pq:
    d, p, r, c = heappop(pq)
    if (d, p) > best[(r, c)]: continue
    if (r, c) == hole: return p
    for ch, dr, dc in [("d",1,0), ("l",0,-1), ("r",0,1), ("u",-1,0)]:
        nr, nc, w = roll(r, c, dr, dc)          # dừng ở tường hoặc ở lỗ
        if w and ((nr, nc) not in best or (d + w, p + ch) < best[(nr, nc)]):
            best[(nr, nc)] = (d + w, p + ch); heappush(pq, (d + w, p + ch, nr, nc))
return "impossible"
```

**Checklist trước khi submit:**

- [ ] Hàm lăn dừng khi **đi qua lỗ**.
- [ ] Khoảng cách = số ô, không phải số lệnh.
- [ ] So sánh `(dist, path)` dạng tuple để phá hòa theo từ điển.
- [ ] Trả về khi **lấy ra** lỗ khỏi heap.
- [ ] Bỏ qua lần lăn có `d = 0`.
- [ ] Trả về đúng chuỗi `"impossible"`.
- [ ] Đã đối chiếu với DFS vét cạn trên mê cung nhỏ ngẫu nhiên.